# 01 — First steps with PySpark

Goal: get comfortable with **SparkSession**, **DataFrames**, **lazy evaluation**, and **Spark SQL**.

Run the cells top to bottom (`Shift+Enter`). While the session is running, open the Spark UI at http://localhost:4040 to see jobs and stages.

## 1. Start Spark

The setup cell does three things you'd normally not need on Linux/Mac:
- points Spark's Python workers at *this* project's `.venv` (otherwise Windows launches the Microsoft Store `python` alias),
- falls back to JDK 17 if `JAVA_HOME` isn't set in this process,
- uses `local[4]` (4 cores) — using all 16 cores at once makes Python workers crash on this machine.

In [1]:
import os, sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ.setdefault("JAVA_HOME", r"C:\Program Files\Java\jdk-17")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[4]")
    .appName("first-steps")
    .config("spark.sql.shuffle.partitions", "4")  # default 200 is overkill locally
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Spark 4.3.0


## 2. Your first DataFrame

A DataFrame is a distributed table with a schema. Here we build one from Python data; later you'll read CSV/Parquet files.

In [2]:
sales = spark.createDataFrame(
    [
        ("2026-01-03", "Athens",       "laptop",  1, 950.0),
        ("2026-01-03", "Thessaloniki", "mouse",   3,  25.0),
        ("2026-01-04", "Athens",       "monitor", 2, 210.0),
        ("2026-01-05", "Patras",       "laptop",  1, 990.0),
        ("2026-01-05", "Athens",       "mouse",   5,  22.0),
        ("2026-01-06", "Thessaloniki", "laptop",  2, 940.0),
        ("2026-01-07", "Patras",       "monitor", 1, 199.0),
        ("2026-01-07", "Athens",       "laptop",  1, 970.0),
    ],
    schema="date STRING, city STRING, product STRING, qty INT, unit_price DOUBLE",
)

sales.printSchema()
sales.show()

root
 |-- date: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- unit_price: double (nullable = true)

+----------+------------+-------+---+----------+
|      date|        city|product|qty|unit_price|
+----------+------------+-------+---+----------+
|2026-01-03|      Athens| laptop|  1|     950.0|
|2026-01-03|Thessaloniki|  mouse|  3|      25.0|
|2026-01-04|      Athens|monitor|  2|     210.0|
|2026-01-05|      Patras| laptop|  1|     990.0|
|2026-01-05|      Athens|  mouse|  5|      22.0|
|2026-01-06|Thessaloniki| laptop|  2|     940.0|
|2026-01-07|      Patras|monitor|  1|     199.0|
|2026-01-07|      Athens| laptop|  1|     970.0|
+----------+------------+-------+---+----------+



## 3. Transformations are lazy, actions run

`withColumn`, `filter`, `select`, `groupBy` only **build a plan**. Nothing executes until an **action** (`show`, `count`, `collect`, `write`).

Run the next cell — it returns instantly and prints only a DataFrame description, no data.

In [3]:
enriched = (
    sales
    .withColumn("date", F.to_date("date"))
    .withColumn("revenue", F.col("qty") * F.col("unit_price"))
    .filter(F.col("revenue") > 100)
)
enriched  # no job ran yet

DataFrame[date: date, city: string, product: string, qty: int, unit_price: double, revenue: double]

In [4]:
enriched.show()           # action -> a job runs (check the Spark UI)
print(enriched.count())   # another action -> another job

+----------+------------+-------+---+----------+-------+
|      date|        city|product|qty|unit_price|revenue|
+----------+------------+-------+---+----------+-------+
|2026-01-03|      Athens| laptop|  1|     950.0|  950.0|
|2026-01-04|      Athens|monitor|  2|     210.0|  420.0|
|2026-01-05|      Patras| laptop|  1|     990.0|  990.0|
|2026-01-05|      Athens|  mouse|  5|      22.0|  110.0|
|2026-01-06|Thessaloniki| laptop|  2|     940.0| 1880.0|
|2026-01-07|      Patras|monitor|  1|     199.0|  199.0|
|2026-01-07|      Athens| laptop|  1|     970.0|  970.0|
+----------+------------+-------+---+----------+-------+

7


## 4. Aggregations (this causes a *shuffle*)

`groupBy` has to bring all rows with the same key to the same partition — data moves between partitions. Shuffles are the main cost in Spark.

In [ ]:
by_city = (
    enriched
    .groupBy("city")
    .agg(
        F.sum("revenue").alias("total_revenue"),
        F.count("*").alias("orders"),
        F.round(F.avg("revenue"), 2).alias("avg_order"),
    )
    .orderBy(F.desc("total_revenue"))
)
by_city.show()

## 5. Look at the plan

`explain()` shows what Spark will actually do. Look for `Exchange` — that's the shuffle.

In [ ]:
by_city.explain()

## 6. Same thing in Spark SQL

Register the DataFrame as a temporary view and query it with SQL. Both APIs compile to the same plan.

In [ ]:
enriched.createOrReplaceTempView("sales")

spark.sql("""
    SELECT product,
           SUM(qty)     AS units,
           SUM(revenue) AS revenue
    FROM sales
    GROUP BY product
    ORDER BY revenue DESC
""").show()

## 7. Exercises

Try these yourself (DataFrame API first, then the same in SQL):

1. Total revenue **per day**, sorted by date.
2. Which city sold the most **laptops** (by quantity)?
3. Add a column `price_band` that is `"high"` when `unit_price >= 500`, else `"low"` (hint: `F.when(...).otherwise(...)`), then count orders per band.
4. Run `sales.rdd.getNumPartitions()` and `by_city.rdd.getNumPartitions()` — why are they different?

In [ ]:
# 1. revenue per day


In [ ]:
# 2. top laptop city


In [ ]:
# 3. price_band


In [ ]:
# 4. partitions


## 8. Stop Spark when you're done

In [ ]:
spark.stop()